# Preprocessing
In this notebook, we will create a pipeline which performs the following transformations on a
dataset passed through [`/filter_data.ipynb`](./filter_data.ipynb).

> As found in [`exploratory_data_analysis.ipynb`](./exploratory_data_analysis.ipynb), the agencies
> within **NYC311** feature vastly different scales and values depending on the agency. As such, we 
> will proceed by creating a pipeline for each model individually, set to the specific agencies
> datasets.

> We will fit our transformation pipeline using data from **NYC311**, in 2025.

| Feature | Null Response | Transformation | Notes |
|---|---|---|---|
| `problem` | — | One-hot encoding | |
| `detail`| Impute with "NoValue" | One-hot encoding | |
| `method`| Impute with "NoValue" | One-hot encoding | |
| `borough` | — | One-hot encoding | |
| `zipcode` | — | Target encoding, Standardization | |
| `c_month` | — | Cyclical encoding | Derived from `created` | 
| `c_day` | — | Cyclical encoding | Derived from `created` | 
| `c_hour`| — | Cyclical encoding | Derived from `created` | 


# Setup

In [1]:
# setup
import numpy as np
import pandas as pd

In [2]:
# grab data
df = pd.read_csv('../data/nyc311_2025.csv', index_col='id')

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 3475290 entries, 67351762 to 63577994
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   created          str    
 1   closed           str    
 2   agency_name      str    
 3   problem          str    
 4   detail           str    
 5   borough          str    
 6   lat              float64
 7   long             float64
 8   method           str    
 9   zipcode          float64
 10  resolution_time  float64
dtypes: float64(4), str(7)
memory usage: 318.2 MB


## Extracting Agencies

In [4]:
# extract agency names
agency_names = df.agency_name.unique().tolist()
agency_names

['New York City Police Department',
 'Taxi and Limousine Commission',
 'Department of Housing Preservation and Development',
 'Department of Buildings',
 'Department of Transportation',
 'Department of Sanitation',
 'Department of Consumer and Worker Protection',
 'Department of Environmental Protection',
 'Department of Parks and Recreation',
 'Department of Health and Mental Hygiene',
 'Office of the Sheriff',
 'Department of Homeless Services',
 'Department of Education',
 'Office of Technology and Innovation',
 'Economic Development Corporation']

In [6]:
# separate df into agencies
agency_df = {}

for agency in agency_names:
    agency_df[agency] = df[df.agency_name == agency]

In [16]:
# sanity check
res1 = df[df.agency_name == ''] # empty clone
for key in agency_df.keys():
    res1 = pd.concat([res1, agency_df[key].head(1)])

res1

,created,closed,agency_name,problem,detail,borough,lat,long,method,zipcode,resolution_time
id,,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,10029.0,0.684
67345192,2025-12-31 23:58:25,2026-01-02 11:55:46,Taxi and Limousine Commission,Lost Property,Bag/Wallet,MANHATTAN,40.738077,-73.992123,PHONE,10011.0,35.956
67350845,2025-12-31 23:57:56,2026-01-03 22:59:48,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,BROOKLYN,40.600602,-73.933754,PHONE,11229.0,71.031
67353895,2025-12-31 23:57:09,2026-03-02 00:00:00,Department of Buildings,Boilers,Boiler - Defective/Inoperative/No Permit,MANHATTAN,40.874848,-73.910831,UNKNOWN,10463.0,1440.048
67344470,2025-12-31 23:57:00,2026-01-09 01:53:00,Department of Transportation,Street Light Condition,Street Light Out,BRONX,40.871462,-73.830537,UNKNOWN,10475.0,193.933
67347694,2025-12-31 23:51:04,2026-01-02 12:28:44,Department of Sanitation,Illegal Dumping,Removal Request,QUEENS,40.662020,-73.775446,MOBILE,11434.0,36.628
67350123,2025-12-31 23:31:29,2026-02-01 04:01:36,Department of Consumer and Worker Protection,Consumer Complaint,Other Store (Non-Food),MANHATTAN,40.745620,-73.999285,ONLINE,10011.0,748.502
67352233,2025-12-31 23:26:00,2026-01-28 07:20:00,Department of Environmental Protection,Water System,No Water (WNW),STATEN ISLAND,40.642804,-74.078712,PHONE,10301.0,655.900
67349958,2025-12-31 23:16:33,2026-01-05 14:48:05,Department of Parks and Recreation,Damaged Tree,Branch or Limb Has Fallen Down,QUEENS,40.739975,-73.705165,ONLINE,11004.0,111.526


# Pipeline Creation
To ensure an easy process when defining our pipeline, we will first create a pipeline that transforms
only **NYPD** data. After our pipeline is created, we can fit and apply the changes to each and every
other agency as well.

In [17]:
df = agency_df['New York City Police Department']
df.head()

,created,closed,agency_name,problem,detail,borough,lat,long,method,zipcode,resolution_time
id,,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,10029.0,0.684
67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.825137,-73.949447,ONLINE,10031.0,1.072
67346873,2025-12-31 23:59:21,2026-01-01 00:58:13,New York City Police Department,Blocked Driveway,Partial Access,BROOKLYN,40.619995,-73.921167,PHONE,11234.0,0.981
67353004,2025-12-31 23:59:20,2026-01-01 00:57:31,New York City Police Department,Noise - Residential,Loud Music/Party,BROOKLYN,40.591542,-73.955979,ONLINE,11235.0,0.970
67350526,2025-12-31 23:59:12,2026-01-01 00:41:16,New York City Police Department,Noise - Commercial,Loud Music/Party,MANHATTAN,40.822593,-73.949525,MOBILE,10031.0,0.701


## Categorical Variables
Our first transformation will consist of a series of *One-hot encoding* transformations on `problem`,
`detail`, `method`, and `borough`. Note that `detail` and `method` will have their own pipeline, as
we would like to keep (impute) any potential missing data with `"NoValue"`. We ignore fields like
`problem` and `borough` as these are critical fields within the input, and cannot be left blank.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


cat_features = ['problem', 'borough', 'detail', 'method']

cat_pipeline_std = Pipeline([
    ('one-hot', OneHotEncoder(drop='first', sparse_output=False))
])

cat_pipeline_impute = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='NoValue')),
    ('one-hot', OneHotEncoder(drop='first', sparse_output=False))
])

cat_pipeline = ColumnTransformer([
    ('cat_std', cat_pipeline_std, ['problem', 'borough']),
    ('cat_impute', cat_pipeline_impute, ['detail', 'method'])],
    remainder='drop', # there should not be any, in the end we will use a column transformer 
    verbose_feature_names_out=True
)

# test
cat_test = df[cat_features]
cat_pipeline.fit(cat_test)
cat_test_res = cat_pipeline.transform(cat_test)
pd.DataFrame(cat_test_res, columns=cat_pipeline.get_feature_names_out()).head(2)

,cat_std__problem_Animal-Abuse,cat_std__problem_Bike/Roller/Skate,cat_std__problem_Bike/Roller/Skate Chronic,cat_std__problem_Blocked Driveway,cat_std__problem_Disorderly Youth,cat_std__problem_Drinking,cat_std__problem_Drug Activity,cat_std__problem_Encampment,cat_std__problem_Graffiti,cat_std__problem_Illegal Fireworks,...,cat_impute__detail_Underage - Licensed Est,cat_impute__detail_Unspecified,cat_impute__detail_Use Indoor,cat_impute__detail_Use Outside,cat_impute__detail_Vehicle,cat_impute__detail_With License Plate,cat_impute__method_ONLINE,cat_impute__method_OTHER,cat_impute__method_PHONE,cat_impute__method_UNKNOWN
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## Zipcode
One-hot encoding our `zipcode` variable would result with a very large amount of categorical variables
with little information gain, we can instead perform techniques like *target encoding*, to find the
mean resolution time across zipcodes. After we perform *target encoding*, we can *standardize* the
results.